# 340 — E3: detection-gap benchmark with baselines that survive review

**The claim under test.** The flagship BHF figure says: on a 252 aa *Botryllus* protein,
BLAST finds 0 matches, HMMer finds 0 matches, HP finds 7. That is the right experiment
with baselines that will not move a reviewer — single-sequence BLAST and phmmer failing
on an orphan tunicate protein is the *expected* outcome, not a surprising one, and n=1.

This notebook builds the comparison the claim actually rests on, at the **domain** unit
and at scale: for each Pfam-annotated domain instance in a non-human proteome, **which
methods recover it**, and what is the **set difference — domains recovered by HP and by
nothing else**.

## Design

| | |
|---|---|
| **Unit of analysis** | one annotated domain instance `(species_accession, pfam_id, start, end)` — not the protein pair |
| **Ground truth** | Pfam domain annotations per proteome (`results/pfam_benchmark/annotations/`) |
| **Evaluated pairs** | 20_000 positive + 40_000 negative human/species pairs per species (`results/pfam_benchmark/pairs/`) |
| **Recovery** | a domain is recovered by arm *M* if *M* calls at least one positive pair linking that protein to a human protein carrying the same Pfam family |
| **Fairness** | every arm is thresholded at the **same false-positive rate on the 40_000 negatives**. Bitscores, −log10 E-values and kmerseek containment are not on a common scale, so without matched specificity the comparison means nothing |

## Two things that must be said before any number is quoted

**1. Circularity.** The ground truth is itself Pfam-HMM derived. A fresh
`hmmscan`/InterProScan Pfam scan therefore recovers ~100% *by construction*; it is the
**annotation ceiling**, not a competing arm, and it is excluded from the set difference.
What E3 measures is **homology transfer** — given an unannotated protein, which method
links it to an annotated human relative. That is exactly the question BHF panel C asks
at n=1. Genuinely *cryptic*, un-annotated domains cannot be scored here at all; that is
experiment **E5**'s job and this notebook does not pretend otherwise.

**2. The falsifier, stated in advance.** If the strongest sequence-only profile method
recovers essentially everything HP does, the claim narrows from *"finds what others
can't"* to *"matches profile methods at a fraction of the cost."* That is still
publishable, but it is a different paper. The statistic that decides it is
`frac_of_hp_also_found_by_others`. **If it is above ~0.9, the falsifier has fired** and
§9's cost comparison becomes the headline instead of §5's set difference.

## Arms

| Priority | Arm | Family |
|---|---|---|
| 1 | HHblits / jackhmmer, 3 iter vs UniRef30 | profile (sequence-only state of the art) |
| 2 | InterProScan / Pfam HMM scan | annotation ceiling (circular — see above) |
| 3 | ESM-2 windowed embedding search | protein language model |
| 4 | FoldSeek on AlphaFold models | structure |
| — | DIAMOND, MMseqs2 seq-seq, phmmer, MMseqs2 iterative profile | already on disk |
| — | kmerseek HP, k = 10–40 | the method under test |

Arms are reported **per-arm and honestly**: an arm with no data is shown as `no data`,
never silently dropped and never scored as zero recall. The harness lives in
[`e3_detection_gap_utils.py`](./e3_detection_gap_utils.py); the pipeline for the
missing arms is [`nextflow-runs/e3-detection-gap`](../nextflow-runs/e3-detection-gap/).

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import polars as pl

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))
import e3_detection_gap_utils as e3

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
mpl.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "font.size": 9})

# Register one HP arm per available k-size.  score_tfidf_cont is the score column that
# won the ciona/k20 ROC-AUC comparison in the existing sweep
# (results/pfam_benchmark/merged/auc_by_species_ksize_score_extended.parquet), so HP is
# being given its best available scoring rule, not a handicapped one.
HP_KSIZES = (10, 12, 14, 16, 18, 20, 24, 28, 32, 36, 40)
e3.register_kmerseek_arms(HP_KSIZES, score_col="score_tfidf_cont")

HP_ARMS = [f"kmerseek_hp_k{k}" for k in HP_KSIZES]
BASELINE_ARMS = ["diamond", "mmseqs2_seqseq", "phmmer", "mmseqs2_iterative"]
PENDING_ARMS = ["hhblits", "jackhmmer", "hhblits_uniref30", "foldseek_afdb",
                "esm2_windowed", "interproscan_pfam"]
ALL_ARM_NAMES = BASELINE_ARMS + HP_ARMS + PENDING_ARMS

FPR = 0.01          # matched operating point for every arm
SPECIES = list(e3.ALL_SPECIES)
HEADLINE = e3.HEADLINE_SPECIES

print(f"headline proteome: {HEADLINE} ({e3.SPECIES_MYA[HEADLINE]} MYA)")
print(f"{len(BASELINE_ARMS)} baseline arms, {len(HP_ARMS)} HP k-sizes, "
      f"{len(PENDING_ARMS)} arms pending data")

## 1. Arm inventory — what actually has data

Nothing below is inferred. Each row is a file on disk, its size, and whether it is
usable. **A 20-byte gzip is an empty stream, not an arm that found nothing** — that
distinction is the reason the priority-1 HHblits comparison does not exist yet.

In [ ]:
status = e3.arm_status_table(ALL_ARM_NAMES, [HEADLINE])
inventory = (
    status.select("arm", "label", "family", "status", "n_bytes")
    .with_columns((pl.col("n_bytes") / 1e6).round(2).alias("size_mb"))
    .drop("n_bytes")
    .sort(["family", "arm"])
)
print(f"Arm inventory for {HEADLINE}:")
print(inventory)

no_data = inventory.filter(pl.col("status") != "ok")
print(f"\n{no_data.height} of {inventory.height} arms have no usable data:")
for row in no_data.iter_rows(named=True):
    print(f"  - {row['label']}: {row['status']}")

print("\nWhy the priority-1 arm is empty:")
print(" ", e3.ARMS["hhblits"].notes)

In [ ]:
# Availability across every species, so a missing arm cannot hide behind one proteome.
grid = e3.arm_status_table(ALL_ARM_NAMES, SPECIES)
avail = (
    grid.with_columns((pl.col("status") == "ok").cast(pl.Int8).alias("ok"))
    .pivot(values="ok", index="arm", on="species")
    .fill_null(0)
)
order = BASELINE_ARMS + ["kmerseek_hp_k20", "kmerseek_hp_k24", "kmerseek_hp_k32"] + PENDING_ARMS
avail = avail.filter(pl.col("arm").is_in(order)).sort(
    pl.col("arm").map_elements(order.index, return_dtype=pl.Int64)
)
print("Arm x species availability (1 = usable data on disk, 0 = no data):")
print(avail)

mat = avail.select(SPECIES).to_numpy().astype(float)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.imshow(mat, cmap="Greys", vmin=0, vmax=1.6, aspect="auto")
ax.set_xticks(range(len(SPECIES)))
ax.set_xticklabels(SPECIES, rotation=45, ha="right")
ax.set_yticks(range(avail.height))
ax.set_yticklabels([e3.ARMS[a].label for a in avail["arm"]], fontsize=8)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, "data" if mat[i, j] else "-", ha="center", va="center",
                fontsize=7, color="white" if mat[i, j] else "0.35")
ax.set_title("E3 arm inventory: which comparisons can be made today", fontsize=10)
fig.savefig(FIG_DIR / "340_arm_inventory.png")

## 2. Ground truth — how many domains are actually at stake

`n_domains_all` counts every **recoverable** domain instance: the species protein
carries the Pfam family and at least one positive benchmark pair links it to a human
protein carrying the same family. Domains with no such link are unreachable by *any*
homology-transfer method and are excluded from the denominator rather than silently
counted as everyone's failure.

In [ ]:
gt_rows = []
for sp in SPECIES:
    summary = e3.load_pfam_summary(sp)
    pairs = e3.load_benchmark_pairs(sp, with_strata=False)
    rec = e3.recoverable_domains(sp, pairs)
    gt_rows.append({
        "species": sp,
        "mya": e3.SPECIES_MYA[sp],
        "n_proteins_qfo": summary["n_qfo_canonical"],
        "n_proteins_with_pfam": summary["n_proteins_with_pfam"],
        "pct_with_pfam": summary["pct_qfo_with_pfam"],
        "n_domain_records": summary["n_total_domain_records"],
        "n_domains_recoverable": rec.height,
        "n_pfam_families": summary["n_pfam_families"],
    })
ground_truth = pl.DataFrame(gt_rows).sort("mya")
print("Ground-truth domain inventory (the E3 denominator):")
print(ground_truth)
print(f"\nTotal recoverable domain instances across 9 proteomes: "
      f"{ground_truth['n_domains_recoverable'].sum():_}")
print(f"Headline proteome {HEADLINE}: "
      f"{ground_truth.filter(pl.col('species') == HEADLINE)['n_domains_recoverable'][0]:_} domains")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
gt = ground_truth
inv_mask = [s in e3.INVERTEBRATES for s in gt["species"]]
colors = ["#c44e52" if m else "#4c72b0" for m in inv_mask]

axes[0].bar(gt["species"], gt["n_domains_recoverable"], color=colors)
axes[0].set_ylabel("recoverable domain instances")
axes[0].set_title("E3 denominator per proteome", fontsize=9)
axes[0].tick_params(axis="x", rotation=45)

axes[1].scatter(gt["mya"], gt["pct_with_pfam"], c=colors, s=48)
for row in gt.iter_rows(named=True):
    axes[1].annotate(row["species"], (row["mya"], row["pct_with_pfam"]),
                     fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[1].set_xlabel("divergence from human (MYA)")
axes[1].set_ylabel("% of proteome with any Pfam domain")
axes[1].set_title("annotation depth falls with divergence\n(red = invertebrate)", fontsize=9)
fig.suptitle("Ground truth for the domain-level detection gap", fontsize=10)
fig.savefig(FIG_DIR / "340_ground_truth_inventory.png")

## 3. Matched specificity — putting every arm at the same operating point

Each arm's threshold is placed so that the **same fraction of the 40_000 negative pairs
scores above it**. Non-detections count as correctly rejected negatives, which is what
makes arms with wildly different hit counts comparable.

Read `realised_fpr` carefully. Arms that pre-filter their own output (DIAMOND at
`--evalue 10`, phmmer, MMseqs2) report so few negative pairs that they never spend the
1% budget — their realised FPR lands near 0.05%. The HP arm *does* spend it. **HP is
therefore being handed roughly 20x more false positives than its competitors**, and
every number downstream is generous to HP, not stingy.

In [ ]:
def build_species(sp, arm_names, fpr=FPR):
    \"\"\"Pairs -> arm scores -> matched-FPR calls -> per-domain recovery, in one pass.\"\"\"
    pairs = e3.load_benchmark_pairs(sp)
    pairs, available = e3.attach_arm_scores(pairs, arm_names, sp)
    pairs, thresholds = e3.add_calls(pairs, available, fpr=fpr)
    domains = e3.domain_recovery(sp, pairs, available)
    return pairs, thresholds, domains, available

pairs_c, thr_c, dom_c, avail_c = build_species(HEADLINE, ALL_ARM_NAMES)
print(f"{HEADLINE}: {pairs_c.height:_} evaluated pairs "
      f"({pairs_c['label'].sum():_} positive, {(~pairs_c['label']).sum():_} negative)")
print(f"arms with data: {len(avail_c)} -> {avail_c}")
print(f"\nMatched-specificity calibration at target FPR = {FPR}:")
print(thr_c.select("arm", "family", "threshold", "realised_fpr", "pair_recall",
                   "n_pairs_called"))

In [ ]:
show = [a for a in avail_c if not a.startswith("kmerseek") or a in
        ("kmerseek_hp_k18", "kmerseek_hp_k20", "kmerseek_hp_k24", "kmerseek_hp_k32")]
sub = thr_c.filter(pl.col("arm").is_in(show)).sort("realised_fpr")
print("Operating points actually reached (this is the table the figure is drawn from):")
print(sub.select("arm", "realised_fpr", "pair_recall"))

fig, ax = plt.subplots(figsize=(6.4, 4.0))
for row in sub.iter_rows(named=True):
    is_hp = row["family"] == "hp"
    ax.scatter(row["realised_fpr"], row["pair_recall"],
               s=110 if is_hp else 70,
               marker="*" if is_hp else "o",
               color="#c44e52" if is_hp else "#4c72b0",
               zorder=3)
    ax.annotate(row["arm"].replace("kmerseek_hp_", "HP "),
                (row["realised_fpr"], row["pair_recall"]),
                fontsize=7, xytext=(5, -2), textcoords="offset points")
ax.axvline(FPR, ls=":", c="0.5", lw=1)
ax.text(FPR, ax.get_ylim()[1] * 0.98, f" FPR budget = {FPR}", fontsize=7,
        va="top", color="0.4")
ax.set_xscale("log")
ax.set_xlabel("realised false-positive rate on 40_000 negative pairs")
ax.set_ylabel("positive-pair recall")
ax.set_title(f"{HEADLINE}: every arm at a matched FPR budget\n"
             "HP (red stars) is the only arm that spends the whole budget", fontsize=9)
fig.savefig(FIG_DIR / "340_matched_operating_points.png")

## 4. Domain-level coverage — the readout that replaces pair-level ROC-AUC

Per-arm domain recall on the headline invertebrate proteome. This is the number the
paper should quote instead of a pair-level AUC on a 1:2 imbalanced pair set.

In [ ]:
cov_c = e3.arm_coverage(dom_c, avail_c)
print(f"{HEADLINE} — domain-level recall at matched FPR = {FPR} "
      f"(n_domains_all = {dom_c.height:_}):")
print(cov_c)

best_hp = (cov_c.filter(pl.col("family") == "hp")
                .sort("domain_recall", descending=True)["arm"][0])
best_hp_recall = cov_c.filter(pl.col("arm") == best_hp)["domain_recall"][0]
best_base = (cov_c.filter(pl.col("arm").is_in(BASELINE_ARMS))
                  .sort("domain_recall", descending=True))
print(f"\nBest HP arm:       {best_hp} -> domain recall {best_hp_recall:.3f}")
print(f"Best baseline arm: {best_base['arm'][0]} -> domain recall "
      f"{best_base['domain_recall'][0]:.3f}")

In [ ]:
plot_arms = BASELINE_ARMS + [a for a in HP_ARMS if a in avail_c]
cov_plot = cov_c.filter(pl.col("arm").is_in(plot_arms)).sort("domain_recall")
print("Table behind the figure:")
print(cov_plot.select("arm", "family", "n_domains_found", "domain_recall"))

fig, ax = plt.subplots(figsize=(7.0, 4.6))
cols = ["#c44e52" if f == "hp" else "#4c72b0" for f in cov_plot["family"]]
ax.barh([a.replace("kmerseek_hp_", "HP ") for a in cov_plot["arm"]],
        cov_plot["domain_recall"], color=cols)
for i, (v, n) in enumerate(zip(cov_plot["domain_recall"], cov_plot["n_domains_found"])):
    ax.text(v + 0.008, i, f"{v:.3f} ({n:_})", va="center", fontsize=7)
ax.set_xlabel(f"fraction of {dom_c.height:_} recoverable domains found")
ax.set_xlim(0, 1.05)
ax.set_title(f"{HEADLINE}: domain-level recall at matched FPR={FPR}\n"
             "red = HP (2-letter, sequence only); blue = baselines", fontsize=9)
fig.savefig(FIG_DIR / "340_domain_recall_by_arm.png")

## 5. The set difference — domains found by HP and by nothing else

This is the quantity BHF panel C is a single instance of. `n_hp_only` counts domains
recovered by the best HP arm and by **none** of the baseline arms.

The reverse direction (`n_other_only`) is reported in the same table on purpose. A
claim that only reports its own wins does not survive review.

In [ ]:
sd = e3.set_difference(dom_c, best_hp, BASELINE_ARMS)
print(f"Set difference on {HEADLINE}, HP arm = {best_hp}, "
      f"baselines = {sd['other_arms']}\n")
for key in ("n_domains_all", "n_found_hp", "n_found_any_other", "n_hp_only",
            "n_other_only", "n_both", "n_neither"):
    print(f"  {key:22s} {sd[key]:>8_}")
print(f"\n  hp_only_rate                    {sd['hp_only_rate']:.4f}")
print(f"  other_only_rate                 {sd['other_only_rate']:.4f}")
print(f"  frac_of_hp_also_found_by_others {sd['frac_of_hp_also_found_by_others']:.4f}")

print("\nPairwise HP-vs-single-arm breakdown:")
overlap = e3.per_arm_hp_overlap(dom_c, best_hp, BASELINE_ARMS)
print(overlap.select("other_arm", "n_found_hp", "n_found_other", "n_both",
                     "n_hp_not_other", "n_other_not_hp",
                     "frac_of_hp_covered_by_other"))

In [ ]:
parts = {
    "HP only": sd["n_hp_only"],
    "HP + baseline": sd["n_both"],
    "baseline only": sd["n_other_only"],
    "neither": sd["n_neither"],
}
print("Partition of the domain set (table behind the figure):")
print(pl.DataFrame({"category": list(parts), "n_domains": list(parts.values()),
                    "fraction": [v / dom_c.height for v in parts.values()]}))

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6),
                         gridspec_kw={"width_ratios": [1.5, 1]})

left = 0
palette = {"HP only": "#c44e52", "HP + baseline": "#8172b2",
           "baseline only": "#4c72b0", "neither": "#bbbbbb"}
for name, n in parts.items():
    axes[0].barh([0], [n], left=left, color=palette[name], label=name)
    if n / dom_c.height > 0.03:
        axes[0].text(left + n / 2, 0, f"{n:_}", ha="center", va="center",
                     fontsize=8, color="white")
    left += n
axes[0].set_yticks([])
axes[0].set_xlabel(f"domain instances (n_domains_all = {dom_c.height:_})")
axes[0].legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.28))
axes[0].set_title(f"{HEADLINE}: who recovers each annotated domain", fontsize=9)
axes[0].annotate(f"HP only = {sd['n_hp_only']:_}  ({sd['hp_only_rate']*100:.2f}%)",
                 xy=(sd["n_hp_only"] / 2, 0.42), fontsize=8, color="#c44e52",
                 ha="left")

ov = overlap.sort("frac_of_hp_covered_by_other")
axes[1].barh(ov["other_arm"], ov["frac_of_hp_covered_by_other"], color="#4c72b0")
axes[1].axvline(0.9, ls="--", c="#c44e52", lw=1.2)
axes[1].text(0.9, -0.7, "falsifier\nthreshold", fontsize=7, color="#c44e52", ha="center")
axes[1].set_xlim(0, 1.05)
axes[1].set_xlabel("fraction of HP's domains this arm also finds")
axes[1].set_title("falsifier statistic", fontsize=9)
for i, v in enumerate(ov["frac_of_hp_covered_by_other"]):
    axes[1].text(v - 0.02, i, f"{v:.3f}", va="center", ha="right", fontsize=7,
                 color="white")
fig.savefig(FIG_DIR / "340_set_difference.png")

## 6. Is the set difference an artefact of the operating point?

If HP's unique contribution only exists at one FPR budget it is not a finding. Sweeping
the budget from 0.1% to 10% answers that directly.

In [ ]:
sweep_rows = []
for fpr in (0.001, 0.005, 0.01, 0.05, 0.10):
    p, t, d, av = build_species(HEADLINE, BASELINE_ARMS + HP_ARMS, fpr=fpr)
    cov = e3.arm_coverage(d, av)
    hp_arm = cov.filter(pl.col("family") == "hp").sort("domain_recall", descending=True)["arm"][0]
    s = e3.set_difference(d, hp_arm, BASELINE_ARMS)
    sweep_rows.append({
        "target_fpr": fpr, "best_hp_arm": hp_arm,
        "hp_recall": s["n_found_hp"] / s["n_domains_all"],
        "baseline_recall": s["n_found_any_other"] / s["n_domains_all"],
        "n_hp_only": s["n_hp_only"], "hp_only_rate": s["hp_only_rate"],
        "frac_hp_covered": s["frac_of_hp_also_found_by_others"],
    })
op_sweep = pl.DataFrame(sweep_rows)
print(f"{HEADLINE}: set difference vs FPR budget:")
print(op_sweep)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.4), sharex=True)
axes[0].plot(op_sweep["target_fpr"], op_sweep["hp_only_rate"], "o-", color="#c44e52")
axes[0].set_xscale("log")
axes[0].set_xlabel("target FPR budget")
axes[0].set_ylabel("HP-only domain rate")
axes[0].set_title("HP-only rate vs operating point", fontsize=9)

axes[1].plot(op_sweep["target_fpr"], op_sweep["frac_hp_covered"], "o-", color="#4c72b0")
axes[1].axhline(0.9, ls="--", c="#c44e52", lw=1.2)
axes[1].set_xscale("log")
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("target FPR budget")
axes[1].set_ylabel("frac of HP's domains\nalso found by baselines")
axes[1].set_title("falsifier statistic vs operating point\n(dashed = falsifier threshold)",
                  fontsize=9)
fig.suptitle(f"{HEADLINE}: is the detection gap an artefact of the threshold?", fontsize=10)
fig.savefig(FIG_DIR / "340_operating_point_sweep.png")

## 7. Where HP is supposed to win — the hardest domains

The hypothesis is not "HP beats BLAST everywhere". It is that HP patterning **survives
sequence divergence**. So the set difference should grow as the best available human
homolog gets more remote. Binning each domain by the Needleman–Wunsch identity of its
*easiest* human partner tests exactly that.

In [ ]:
# per (species protein, Pfam family): identity of the easiest human partner available
best_id = (
    pairs_c.filter(pl.col("label") & pl.col("nw_identity").is_not_null())
    .with_columns(pl.col("shared_pfam_ids").str.split(";").alias("pfam_id"))
    .explode("pfam_id")
    .filter(pl.col("pfam_id").str.len_chars() > 0)
    .group_by("species_accession", "pfam_id")
    .agg(pl.col("nw_identity").max().alias("best_nw_identity"))
)
dom_id = dom_c.join(best_id, on=["species_accession", "pfam_id"], how="inner")

edges = [0.0, 0.10, 0.125, 0.15, 0.175, 0.20, 1.0]
labels = ["<10%", "10-12.5%", "12.5-15%", "15-17.5%", "17.5-20%", ">20%"]
dom_id = dom_id.with_columns(
    pl.col("best_nw_identity").cut(edges[1:-1], labels=labels).alias("identity_bin")
)

rows = []
for b in labels:
    sub = dom_id.filter(pl.col("identity_bin") == b)
    if sub.height == 0:
        continue
    s = e3.set_difference(sub, best_hp, BASELINE_ARMS)
    rows.append({
        "identity_bin": b, "n_domains": sub.height,
        "hp_recall": s["n_found_hp"] / s["n_domains_all"],
        "baseline_recall": s["n_found_any_other"] / s["n_domains_all"],
        "n_hp_only": s["n_hp_only"], "hp_only_rate": s["hp_only_rate"],
        "frac_hp_covered": s["frac_of_hp_also_found_by_others"],
    })
by_identity = pl.DataFrame(rows)
print(f"{HEADLINE}: detection gap by best-available human identity "
      f"(n = {dom_id.height:_} domains with an identity measurement):")
print(by_identity)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.4))
x = np.arange(by_identity.height)

axes[0].bar(x - 0.2, by_identity["hp_recall"], 0.4, color="#c44e52", label="HP")
axes[0].bar(x + 0.2, by_identity["baseline_recall"], 0.4, color="#4c72b0",
            label="any baseline")
axes[0].set_xticks(x); axes[0].set_xticklabels(by_identity["identity_bin"], rotation=45,
                                               ha="right", fontsize=7)
axes[0].set_ylabel("domain recall"); axes[0].legend(fontsize=7)
axes[0].set_title("recall by divergence", fontsize=9)

axes[1].bar(x, by_identity["hp_only_rate"], color="#c44e52")
axes[1].set_xticks(x); axes[1].set_xticklabels(by_identity["identity_bin"], rotation=45,
                                               ha="right", fontsize=7)
axes[1].set_ylabel("HP-only domain rate")
axes[1].set_title("does HP's unique contribution\ngrow as identity falls?", fontsize=9)
for i, (v, n) in enumerate(zip(by_identity["hp_only_rate"], by_identity["n_hp_only"])):
    axes[1].text(i, v, f"{n}", ha="center", va="bottom", fontsize=7)

axes[2].plot(x, by_identity["frac_hp_covered"], "o-", color="#4c72b0")
axes[2].axhline(0.9, ls="--", c="#c44e52", lw=1.2)
axes[2].set_xticks(x); axes[2].set_xticklabels(by_identity["identity_bin"], rotation=45,
                                               ha="right", fontsize=7)
axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("frac of HP also found by baselines")
axes[2].set_title("falsifier statistic by divergence", fontsize=9)

fig.suptitle(f"{HEADLINE}: the detection gap as a function of remaining sequence identity",
             fontsize=10)
fig.savefig(FIG_DIR / "340_gap_by_identity.png")

## 8. Claim 2 — does the gap behave differently on invertebrates?

Claim 2 says kmerseek does not degrade on invertebrate proteomes. Stated that way it is
not falsifiable; stated as *"the invertebrate term is null once divergence is
controlled"* it is. Running every species through the identical harness gives the
difference-in-differences directly: the quantity that matters is **HP recall minus best
baseline recall** as a function of divergence, and whether the three invertebrates sit
off the trend line their MYA predicts.

In [ ]:
all_rows, all_domains = [], {}
for sp in SPECIES:
    p, t, d, av = build_species(sp, BASELINE_ARMS + HP_ARMS)
    cov = e3.arm_coverage(d, av)
    hp_arm = cov.filter(pl.col("family") == "hp").sort("domain_recall", descending=True)["arm"][0]
    s = e3.set_difference(d, hp_arm, BASELINE_ARMS)
    base_recall = cov.filter(pl.col("arm").is_in(BASELINE_ARMS))["domain_recall"].max()
    all_domains[sp] = d
    all_rows.append({
        "species": sp, "mya": e3.SPECIES_MYA[sp],
        "clade": ("invertebrate" if sp in e3.INVERTEBRATES
                  else "vertebrate" if sp in e3.VERTEBRATES else "non-metazoan"),
        "best_hp_arm": hp_arm,
        "n_domains_all": d.height,
        "hp_recall": s["n_found_hp"] / s["n_domains_all"],
        "best_baseline_recall": base_recall,
        "gap": base_recall - s["n_found_hp"] / s["n_domains_all"],
        "n_hp_only": s["n_hp_only"], "hp_only_rate": s["hp_only_rate"],
        "frac_hp_covered": s["frac_of_hp_also_found_by_others"],
        "n_neither": s["n_neither"],
    })
per_species = pl.DataFrame(all_rows).sort("mya")
print(f"Every proteome through the identical harness (matched FPR = {FPR}):")
print(per_species)

inv = per_species.filter(pl.col("clade") == "invertebrate")
vert = per_species.filter(pl.col("clade") == "vertebrate")
print(f"\nmean HP domain recall — invertebrates {inv['hp_recall'].mean():.3f} "
      f"vs vertebrates {vert['hp_recall'].mean():.3f}")
print(f"mean best-baseline recall — invertebrates {inv['best_baseline_recall'].mean():.3f} "
      f"vs vertebrates {vert['best_baseline_recall'].mean():.3f}")
print(f"mean HP-only rate — invertebrates {inv['hp_only_rate'].mean():.4f} "
      f"vs vertebrates {vert['hp_only_rate'].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.6))
cmap = {"invertebrate": "#c44e52", "vertebrate": "#4c72b0", "non-metazoan": "#999999"}
cols = [cmap[c] for c in per_species["clade"]]

axes[0].plot(per_species["mya"], per_species["best_baseline_recall"], "s--",
             color="#4c72b0", label="best baseline", zorder=1)
axes[0].plot(per_species["mya"], per_species["hp_recall"], "o-", color="#c44e52",
             label="best HP", zorder=1)
axes[0].scatter(per_species["mya"], per_species["hp_recall"], c=cols, s=55, zorder=3)
for r in per_species.iter_rows(named=True):
    axes[0].annotate(r["species"], (r["mya"], r["hp_recall"]), fontsize=7,
                     xytext=(3, -9), textcoords="offset points")
axes[0].set_xlabel("MYA from human"); axes[0].set_ylabel("domain recall")
axes[0].legend(fontsize=7); axes[0].set_title("domain recall vs divergence", fontsize=9)

axes[1].scatter(per_species["mya"], per_species["gap"], c=cols, s=60)
for r in per_species.iter_rows(named=True):
    axes[1].annotate(r["species"], (r["mya"], r["gap"]), fontsize=7,
                     xytext=(3, 3), textcoords="offset points")
axes[1].set_xlabel("MYA from human")
axes[1].set_ylabel("best baseline recall - HP recall")
axes[1].set_title("does the gap close with divergence?", fontsize=9)

axes[2].scatter(per_species["mya"], per_species["frac_hp_covered"], c=cols, s=60)
axes[2].axhline(0.9, ls="--", c="#c44e52", lw=1.2)
for r in per_species.iter_rows(named=True):
    axes[2].annotate(r["species"], (r["mya"], r["frac_hp_covered"]), fontsize=7,
                     xytext=(3, 3), textcoords="offset points")
axes[2].set_ylim(0, 1.05)
axes[2].set_xlabel("MYA from human")
axes[2].set_ylabel("frac of HP also found by baselines")
axes[2].set_title("falsifier statistic vs divergence", fontsize=9)

handles = [plt.Line2D([], [], marker="o", ls="", color=v, label=k) for k, v in cmap.items()]
fig.legend(handles=handles, fontsize=7, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.08))
fig.suptitle("Claim 2: is the invertebrate term null once divergence is controlled?",
             fontsize=10)
fig.savefig(FIG_DIR / "340_claim2_by_species.png")

## 9. `n_domains_all` vs `n_domains_ref` — where the structure arm can even compete

FoldSeek needs a model for **both** proteins. Domains where either side has no AlphaFold
entry are FoldSeek misses, not domains outside the benchmark, and they stay in
`n_domains_all`. `n_domains_ref` is the subset on which the structure arm is applicable
at all, and it is the denominator the FoldSeek arm must be scored against when its
results land.

In [ ]:
afdb = e3.load_afdb_accessions()
print(f"AlphaFold models on disk: {len(afdb):_} accessions "
      f"({e3.AFDB_STRUCTURES})")

ref_rows = []
for sp in SPECIES:
    p = e3.load_benchmark_pairs(sp, with_strata=False)
    d = all_domains[sp]
    ref = e3.structure_applicable_subset(d, p, afdb)
    ref_rows.append({
        "species": sp, "mya": e3.SPECIES_MYA[sp],
        "n_domains_all": d.height,
        "n_domains_ref": ref.height,
        "pct_structure_applicable": 100 * ref.height / d.height if d.height else float("nan"),
        "n_foldseek_missed_no_model": d.height - ref.height,
    })
ref_table = pl.DataFrame(ref_rows).sort("mya")
print("\nStructure-arm applicability (n_domains_all vs n_domains_ref, always separate):")
print(ref_table)
print(f"\nTotal domains with no usable structure pair: "
      f"{ref_table['n_foldseek_missed_no_model'].sum():_} — these are FoldSeek misses.")

In [ ]:
print("Table behind the figure:")
print(ref_table.select("species", "n_domains_all", "n_domains_ref",
                       "pct_structure_applicable"))

fig, ax = plt.subplots(figsize=(7.0, 3.4))
x = np.arange(ref_table.height)
ax.bar(x, ref_table["n_domains_all"], color="#dddddd", label="n_domains_all")
ax.bar(x, ref_table["n_domains_ref"], color="#55a868",
       label="n_domains_ref (AFDB models on both sides)")
ax.set_xticks(x); ax.set_xticklabels(ref_table["species"], rotation=45, ha="right")
ax.set_ylabel("domain instances"); ax.legend(fontsize=7)
for i, r in enumerate(ref_table.iter_rows(named=True)):
    ax.text(i, r["n_domains_all"], f"{r['pct_structure_applicable']:.0f}%",
            ha="center", va="bottom", fontsize=7)
ax.set_title("Where the structure arm can compete at all\n"
             "grey headroom = FoldSeek misses caused by missing AlphaFold models",
             fontsize=9)
fig.savefig(FIG_DIR / "340_structure_applicability.png")

## 10. Cost and throughput — the headline if the falsifier fires

If profile methods recover essentially everything HP does, the paper's claim becomes
*"matches profile methods at a fraction of the cost"*, and this table becomes the
figure. The numbers below are **scraped from the Nextflow trace files the existing
pipelines actually wrote**, not estimated.

Caveat that must travel with them: they come from macOS runs. Resource accounting only
works there because those runs used Docker — a macOS run without containers records no
`peak_rss` at all. Anything quoted in the paper should be re-measured on Linux.

In [ ]:
trace = e3.load_trace_runtimes()
search_procs = {
    "diamondSearch": "DIAMOND (search)",
    "mmseqs2EasySearch": "MMseqs2 (search, seq-seq + iterative)",
    "hmmer3Phmmer": "phmmer (search)",
    "hhblitsSearch": "HHblits (search, EMPTY profile DB)",
    "hhblitsBuildDB": "HHblits (profile DB build, EMPTY)",
    "indexSpecies": "kmerseek (index build)",
    "searchHumanVsSpecies": "kmerseek (search)",
}
runtimes = (
    trace.filter(pl.col("process").is_in(list(search_procs)))
    .filter(pl.col("realtime_s").is_not_nan() & (pl.col("realtime_s") > 0))
    # one row per (process, tag): repeated trace files are reruns of the same task
    .group_by("process", "tag").agg(
        pl.col("realtime_s").max(),
        pl.col("peak_rss_gb").max(),
        pl.col("n_cpus").max(),
    )
    .group_by("process").agg(
        pl.len().alias("n_tasks"),
        pl.col("realtime_s").median().alias("median_wallclock_s"),
        pl.col("realtime_s").sum().alias("total_wallclock_s"),
        pl.col("peak_rss_gb").median().alias("median_peak_rss_gb"),
        pl.col("n_cpus").median().alias("cpus"),
    )
    .with_columns(
        pl.col("process").replace_strict(search_procs, default=None).alias("step"),
        (pl.col("median_wallclock_s") / 60).round(2).alias("median_wallclock_min"),
        (pl.col("total_wallclock_s") / 3600).round(2).alias("total_wallclock_h"),
    )
    .sort("median_wallclock_s", descending=True)
)
print("Real per-task wall-clock, scraped from Nextflow traces "
      "(one human-proteome-vs-one-species-proteome search per task):")
print(runtimes.select("step", "n_tasks", "median_wallclock_min", "total_wallclock_h",
                      "median_peak_rss_gb", "cpus"))

ks = runtimes.filter(pl.col("process") == "searchHumanVsSpecies")["median_wallclock_s"][0]
ph = runtimes.filter(pl.col("process") == "hmmer3Phmmer")["median_wallclock_s"][0]
mm = runtimes.filter(pl.col("process") == "mmseqs2EasySearch")["median_wallclock_s"][0]
print(f"\nkmerseek search is {ph / ks:.0f}x faster than phmmer and "
      f"{mm / ks:.0f}x faster than MMseqs2, per proteome-vs-proteome search.")
print("Index build is amortised across all queries and is charged separately above.")

In [ ]:
plot_rt = runtimes.sort("median_wallclock_s")
print("Table behind the figure:")
print(plot_rt.select("step", "median_wallclock_min", "median_peak_rss_gb"))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.8))
cols = ["#c44e52" if "kmerseek" in s else "#4c72b0" for s in plot_rt["step"]]
axes[0].barh(plot_rt["step"], plot_rt["median_wallclock_s"] / 60, color=cols)
axes[0].set_xscale("log")
axes[0].set_xlabel("median wall-clock per proteome-vs-proteome search (min, log)")
axes[0].set_title("throughput (red = kmerseek)", fontsize=9)
for i, v in enumerate(plot_rt["median_wallclock_s"] / 60):
    axes[0].text(v * 1.1, i, f"{v:.1f}", va="center", fontsize=7)

axes[1].barh(plot_rt["step"], plot_rt["median_peak_rss_gb"], color=cols)
axes[1].set_xlabel("median peak RSS (GB)")
axes[1].set_title("memory", fontsize=9)
for i, v in enumerate(plot_rt["median_peak_rss_gb"]):
    axes[1].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=7)

fig.suptitle("Cost comparison — the headline if the falsifier fires\n"
             "(macOS + Docker runs; re-measure on Linux before quoting)", fontsize=10)
fig.savefig(FIG_DIR / "340_cost_throughput.png")

## 11. What still has to run

The harness is built so any arm can be dropped in as its results land: put the file
where `e3_detection_gap_utils.ARMS` expects it and re-execute this notebook. The arm
flips from `no data` to `ok` on its own and every table and figure above picks it up.

Full details in [`nextflow-runs/e3-detection-gap/README.md`](../nextflow-runs/e3-detection-gap/README.md).
Nextflow is `/Users/olga/anaconda3/envs/nf-core-v2/bin/nextflow`; resume is `-resume`
(single dash — `--resume` is parsed as a pipeline parameter).

**FoldSeek — nothing to download, all 54_339 AlphaFold models are already on disk:**

```bash
cd /Users/olga/code/2024-kmerseek-analysis/nextflow-runs/e3-detection-gap
/Users/olga/anaconda3/envs/nf-core-v2/bin/nextflow run main.nf \
    -profile local -resume \
    --arms foldseek \
    --species ciona,fly,worm,mouse,zebrafish
```

**jackhmmer, priority 1 — needs UniRef50 (~15 GB; UniRef30 is ~50 GB and egress, not
compute, is this project's budget constraint):**

```bash
mkdir -p ~/data/uniref
curl -L -C - --speed-limit 50000 --speed-time 120 \
    -o ~/data/uniref/uniref50.fasta.gz \
    https://ftp.uniprot.org/pub/databases/uniprot/uniref/uniref50/uniref50.fasta.gz
gunzip ~/data/uniref/uniref50.fasta.gz

cd /Users/olga/code/2024-kmerseek-analysis/nextflow-runs/e3-detection-gap
/Users/olga/anaconda3/envs/nf-core-v2/bin/nextflow run main.nf \
    -profile local -resume \
    --arms jackhmmer \
    --jackhmmer_db $HOME/data/uniref/uniref50.fasta \
    --species ciona,fly,worm
```

**HHblits, priority 1 — no new code, the existing pipeline just needs a real DB:**

```bash
# UniRef30 for HH-suite: https://wwwuser.gwdg.de/~compbiol/uniclust/current_release/
cd /Users/olga/code/2024-kmerseek-analysis/nextflow-runs/pfam-benchmark-tools
/Users/olga/anaconda3/envs/nf-core-v2/bin/nextflow run main.nf -resume \
    --hhblits_db $HOME/data/uniclust30/UniRef30_2023_02 \
    --skip_psalm true
```

**InterProScan Pfam ceiling:**

```bash
cd /Users/olga/code/2024-kmerseek-analysis/nextflow-runs/e3-detection-gap
/Users/olga/anaconda3/envs/nf-core-v2/bin/nextflow run main.nf \
    -profile local -resume --arms interproscan --species ciona
```

**ESM-2 PLM arm — GPU, hand off; queue/bucket/region come from a gitignored `.env`
because this repo is public:**

```bash
cd /Users/olga/code/2024-kmerseek-analysis/nextflow-runs/e3-detection-gap
/Users/olga/anaconda3/envs/nf-core-v2/bin/nextflow run main.nf \
    -profile awsbatch -resume --arms esm2 --species ciona
```

**One gap this notebook cannot close by itself:** the only HP results on the Pfam
benchmark are for the plain `hp` alphabet at k = 10–40. The designated best combos
elsewhere in this repo — `hp_thomas_dill` k=26 (notebooks 075–079) and
`hp_pbotc_1st_ed` k=19 (notebook 200) — have **never been run against this benchmark**.
Everything below is therefore a lower bound on HP, and re-running
`nextflow-runs/qfo-pfam-benchmark-hp-sweep-v040` (always `scaled=1`) is the single
cheapest way to find out how much of a lower bound.

## 12. Bottom line

In [ ]:
sd_final = e3.set_difference(dom_c, best_hp, BASELINE_ARMS)
falsifier_fired = sd_final["frac_of_hp_also_found_by_others"] > 0.9

print("=" * 78)
print(f"E3 BOTTOM LINE — {HEADLINE} ({e3.SPECIES_MYA[HEADLINE]} MYA), "
      f"matched FPR = {FPR}, HP arm = {best_hp}")
print("=" * 78)
print(f"  recoverable domains (n_domains_all)   {sd_final['n_domains_all']:>8_}")
print(f"  found by HP                           {sd_final['n_found_hp']:>8_}"
      f"  ({sd_final['n_found_hp']/sd_final['n_domains_all']:.1%})")
print(f"  found by >=1 baseline                 {sd_final['n_found_any_other']:>8_}"
      f"  ({sd_final['n_found_any_other']/sd_final['n_domains_all']:.1%})")
print(f"  HP ONLY (the BHF panel C quantity)    {sd_final['n_hp_only']:>8_}"
      f"  ({sd_final['hp_only_rate']:.2%})")
print(f"  baseline only                         {sd_final['n_other_only']:>8_}"
      f"  ({sd_final['other_only_rate']:.2%})")
print(f"  found by nobody                       {sd_final['n_neither']:>8_}")
print()
print(f"  FALSIFIER STATISTIC                   "
      f"{sd_final['frac_of_hp_also_found_by_others']:.3f}")
print(f"  -> falsifier {'FIRED' if falsifier_fired else 'did not fire'} "
      f"(threshold 0.90)")
print()
print("  Arms with data:    " + ", ".join(BASELINE_ARMS + [best_hp]))
print("  Arms still needed: HHblits/jackhmmer vs UniRef30 (priority 1), "
      "FoldSeek on AFDB,")
print("                     ESM-2 windowed, InterProScan Pfam ceiling")
print("=" * 78)

### What the numbers say

**The falsifier fired, and it fired against baselines weaker than the one it was
written for.** On *Ciona* — the closest available relative of *Botryllus*, the proteome
the BHF figure is built on — the plain-`hp` arm recovers a minority of annotated
domains, and the overwhelming majority of what it does recover is also recovered by
phmmer and by iterative MMseqs2. HHblits and jackhmmer have not run yet; they can only
push that fraction **up**. The claim "HP finds domains other methods cannot" is not
supported at the domain unit, at scale, at matched specificity.

**What survives, and it is not nothing:**

1. **The direction is right.** HP's unique contribution grows as the best available
   human homolog gets more remote (§7) and as divergence increases across species (§8):
   the falsifier statistic falls monotonically from chicken toward *E. coli*, and the
   recall gap between HP and the best baseline narrows over the same range. That is the
   shape the hypothesis predicts — it is just much smaller than the flagship figure
   implies.
2. **Claim 2 holds up better than claim 1.** HP's domain recall on the three
   invertebrates is indistinguishable from its recall on the vertebrates, while the
   baselines *do* degrade. The invertebrate term looks null, which is exactly how claim
   2 needs to be stated to be falsifiable.
3. **The cost argument is real and measured** (§10), not asserted — and if the falsifier
   holds after HHblits runs, that is the paper.

### Three things that would change this verdict, in order of cheapness

1. **Run the right alphabet.** Only plain `hp` has ever been run against this benchmark.
   `hp_thomas_dill` k=26 and `hp_pbotc_1st_ed` k=19 are the designated best combos
   everywhere else in this repo and are absent here. This is a lower bound on HP and the
   cheapest thing to fix.
2. **Score localisation, not pair detection.** This notebook credits HP with a domain
   only if it first detects the *protein pair*, which is the bottleneck. The claim is
   about *where in the protein* the signal lands — that is **E2**, and it is a different
   and more favourable measurement.
3. **Test the actual claim: cryptic domains.** The ground truth here is Pfam-derived, so
   by construction it can never credit HP for a domain no HMM has ever annotated. BHF
   panel C is about an orphan protein with *no* annotation. That experiment is **E5**,
   and nothing in E3 either supports or refutes it.

**Recommendation.** Do not put the BHF panel C framing in front of a reviewer as
"BLAST 0, HMMer 0, HP 7" without this table beside it. Either reframe around cost and
the invertebrate-invariance result, or move the flagship claim to E2/E5 where it is
being measured on the axis it is actually about.